### Step 1: Project Setup

In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY    = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY      = os.getenv("OPENROUTER_API_KEY")

print("Gemini key loaded:       ", "✅" if GEMINI_API_KEY    else "❌ Missing!")
print("Groq key loaded:         ", "✅" if GROQ_API_KEY      else "❌ Missing!")
print("OpenRouter key loaded:   ", "✅" if OPENROUTER_API_KEY      else "❌ Missing!")

Gemini key loaded:        ✅
Groq key loaded:          ✅
OpenRouter key loaded:    ✅


In [2]:
from google import genai
from groq import Groq

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ GEMINI client ready")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ GROQ client ready")

✅ GEMINI client ready
✅ GROQ client ready


### Local PageIndexClient

Passing a LiteLLM model string (instead of "cloud") to index = automatically switches PageIndexClient into local mode and saves all trees into the chosen storage_path directory. Index once, then reuse the saved trees across every agent run and benchmark.

In [3]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from pageindex import PageIndexClient

load_dotenv()

True

#### PageIndex Index via API with rate limits

In [ ]:
# Local mode: pass a dict with "model" and "storage_path" instead of "cloud"
# Model names follow LiteLLM format (e.g., "groq/llama-3.3-70b-versatile")

client = PageIndexClient(
    index={
        "model": "gemini/gemini-3.5-flash-lite",      # lightweight model for node summaries
        "storage_path": "./legal_index",         # local folder storing all JSON trees
    },
    chat="gemini/gemini-3.5-flash-lite",         # stronger model for tree navigation
)

def index_legal_corpus(corpus_dir: str = "./corpus"):

    # Retrieve existing documents
    response = client.list_documents()

    docs = response.get("documents", [])

    # Build existing document registry
    existing = {
        d.get("name"): d.get("doc_id")
        for d in docs
        if isinstance(d, dict)
    }

    registry = {}

    # Preserve existing manifest entries
    manifest_path = Path("./legal_index/manifest.json")

    if manifest_path.exists():
        registry = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )

    # Index PDF files
    for pdf_path in Path(corpus_dir).glob("*.pdf"):

        if pdf_path.name in existing:
            print(f"[SKIP] Already indexed: {pdf_path.name}")

            registry[pdf_path.stem] = existing[pdf_path.name]
            continue

        print(f"[INDEXING] {pdf_path.name} ...")

        result = client.submit_document(str(pdf_path))

        doc_id = result["doc_id"]

        registry[pdf_path.stem] = doc_id

        print(f"[DONE] {pdf_path.name} -> {doc_id}")

    # Save manifest
    Path("./legal_index").mkdir(parents=True, exist_ok=True)

    manifest_path.write_text(
        json.dumps(registry, indent=2),
        encoding="utf-8"
    )

    print("\nSaved manifest to ./legal_index/manifest.json")


# Calling the function
index_legal_corpus()

#### PageIndex Index via Offline LLMs (Ollama)

Because of summary generation of each node, it took more than 8 hrs and still was not able to complete 3 documents.

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from pageindex import PageIndexClient

load_dotenv()
os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")

# Local Ollama for indexing ONLY; zero API calls, zero rate limits
client = PageIndexClient(
    index={
        "model": "ollama_chat/llama3.2:3b",
        # "model": "ollama_chat/qwen2.5:1.5b",
        "storage_path": "./legal_index",
        "optimize": "merge",                # deterministic merge pass (fewer local LLM calls)
        "summary_concurrency": 1,           # prevents Ollama queue timeouts on CPU/laptop GPU
        "summary_max_words": 60,
    }
)

def index_legal_corpus(corpus_dir: str = "./corpus"):
    manifest_dir = Path("./legal_index")
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / "manifest.json"

    response = client.list_documents()
    docs = response.get("documents", []) if isinstance(response, dict) else response
    existing = {
        d.get("name"): d.get("doc_id")
        for d in docs
        if isinstance(d, dict)
    }

    registry = (
        json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest_path.exists()
        else {}
    )

    for pdf_path in sorted(Path(corpus_dir).glob("*.pdf")):
        if pdf_path.name in existing:
            print(f"[SKIP] Already indexed: {pdf_path.name}")
            registry[pdf_path.stem] = existing[pdf_path.name]
            manifest_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
            continue

        print(f"[INDEXING] {pdf_path.name} (local Ollama) ...")
        result = client.submit_document(str(pdf_path))
        doc_id = result["doc_id"]
        registry[pdf_path.stem] = doc_id

        # Save after every document so progress is never lost
        manifest_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
        print(f"[DONE] {pdf_path.name} -> {doc_id}")

    print("\nSaved manifest to ./legal_index/manifest.json")

if __name__ == "__main__":
    index_legal_corpus()

[SKIP] Already indexed: bnsAct2023.pdf
[SKIP] Already indexed: bnssAct2023.pdf
[INDEXING] constitutionOfIndia1950.pdf (local Ollama) ...


### PageIndex's underlying Flash function without LLM summary

Instead of running PageIndexClient.submit_document() (which forces LLM summarization on every node), call PageIndex's underlying Flash function directly with summary=False. Using optimize="merge" (deterministic, LLM-free pass) or optimize=False builds the entire hierarchical tree purely from the PDF layout and bookmarks in seconds—with zero Ollama calls and zero API rate limits.

In [ ]:
import json
import time
from pathlib import Path
import fitz
import pymupdf
from pageindex.flash import page_index_flash


def build_fast_legal_index(corpus_dir: str = "./corpus", output_dir: str = "./legal_index",):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    manifest = {}

    for pdf_path in sorted(Path(corpus_dir).glob("*.pdf")):
        t0 = time.perf_counter()
        print(f"[INDEXING] {pdf_path.name} (zero-LLM Flash mode) ...")

        # Pure deterministic layout/bookmark tree - zero LLM calls
        tree = page_index_flash(
            str(pdf_path),
            summary=False,
            optimize="merge",  # use optimize=False if raw unmerged nodes is needed
        )

        # Save hierarchical tree JSON
        tree_file = out / f"{pdf_path.stem}_tree.json"
        tree_file.write_text(json.dumps(tree, indent=2, ensure_ascii=False), encoding="utf-8")

        # Extract & cache 1-indexed verbatim pages once for <5ms retrieval in LangGraph
        doc = fitz.open(pdf_path)
        pages = {str(i + 1): page.get_text("text") for i, page in enumerate(doc)}
        doc.close()

        pages_file = out / f"{pdf_path.stem}_pages.json"
        pages_file.write_text(json.dumps(pages, indent=2, ensure_ascii=False), encoding="utf-8")

        manifest[pdf_path.stem] = {
            "source_pdf": pdf_path.name,
            "page_count": len(pages),
            "tree_file": str(tree_file),
            "pages_file": str(pages_file),
        }
        print(f"[DONE] {pdf_path.name} ({len(pages)} pages) in {time.perf_counter() - t0:.2f}s")

    (out / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"\nAll statutes indexed. Manifest saved to {out / 'manifest.json'}")


if __name__ == "__main__":
    build_fast_legal_index()

In [1]:
import json
from pathlib import Path
from langchain_core.tools import tool

INDEX_DIR = Path("./legal_index")
MANIFEST = json.loads((INDEX_DIR / "manifest.json").read_text(encoding="utf-8"))

@tool
def list_indexed_statutes() -> str:
    """List all indexed Indian statutes and their page counts."""
    return json.dumps(
        {k: {"file": v["source_pdf"], "pages": v["page_count"]} for k, v in MANIFEST.items()},
        indent=2,
    )

@tool
def get_statute_toc_tree(statute_key: str) -> str:
    """Return the deterministic hierarchical ToC tree (titles and page ranges) for a statute."""
    tree_path = Path(MANIFEST[statute_key]["tree_file"])
    return tree_path.read_text(encoding="utf-8")

@tool
def fetch_verbatim_statute_pages(statute_key: str, start_page: int, end_page: int) -> str:
    """Deterministically fetch exact verbatim statutory text for a 1-indexed page range."""
    pages = json.loads(Path(MANIFEST[statute_key]["pages_file"]).read_text(encoding="utf-8"))
    chunks = [
        f"--- PAGE {p} ---\n{pages[str(p)]}"
        for p in range(start_page, end_page + 1)
        if str(p) in pages
    ]
    return "\n\n".join(chunks)